# Employment by Age (Latest Month)

KOSIS 연령계층별 취업자 데이터를 불러와 최신 월 기준 분포를 시각화합니다.

In [ ]:
from pathlib import Path
import re

import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd

# Anaconda/Jupyter의 현재 작업 폴더와 무관하게 이 프로젝트를 직접 가리킵니다.
REPO_ROOT = Path("/Users/choewonbin/Desktop/socialInteli/Projct")
if not (REPO_ROOT / "employment_wage").exists() or not (REPO_ROOT / "data").exists():
    raise FileNotFoundError(f"프로젝트 루트를 찾을 수 없습니다: {REPO_ROOT}")


def set_korean_font() -> None:
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in ["AppleGothic", "Apple SD Gothic Neo", "Arial Unicode MS", "Malgun Gothic", "Nanum Gothic", "NanumGothic", "DejaVu Sans"]:
        if name in installed:
            matplotlib.rcParams["font.family"] = name
            break
    matplotlib.rcParams["axes.unicode_minus"] = False


def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    for enc in ["cp949", "euc-kr", "utf-8-sig", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("csv", b"", 0, 1, f"Failed to decode: {path}")


ew_dir = REPO_ROOT / "employment_wage"
csv_path = REPO_ROOT / "data" / "raw" / "employment_wage" / "employment_region_age.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"CSV 파일이 없습니다: {csv_path}")

print(f"Using CSV: {csv_path}")
df = read_csv_with_fallback(csv_path)


In [ ]:
age_col = "연령계층별(1)"

if "시도별(1)" in df.columns:
    df = df[df["시도별(1)"] == "계"].copy()

df = df[df[age_col] != "계"].copy()
month_cols = [c for c in df.columns if re.fullmatch(r"\d{4}\.\d{2}", str(c))]
if not month_cols:
    raise ValueError("YYYY.MM 형식 월 컬럼이 없습니다")
latest_col = sorted(month_cols)[-1]

df[latest_col] = pd.to_numeric(df[latest_col], errors="coerce")
plot_df = df[[age_col, latest_col]].dropna().copy()

canonical_ages = ["15 - 19세", "20 - 29세", "30 - 39세", "40 - 49세", "50 - 59세", "60세이상"]
if set(canonical_ages).issubset(set(plot_df[age_col])):
    plot_df = plot_df[plot_df[age_col].isin(canonical_ages)].copy()

def age_sort_key(label: str) -> int:
    m = re.search(r"(\d+)\s*-\s*(\d+)세", label)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d+)세\s*이상", label)
    if m:
        return int(m.group(1))
    return 999

plot_df = plot_df.sort_values(by=age_col, key=lambda s: s.map(age_sort_key))
plot_df


In [ ]:
set_korean_font()

plt.figure(figsize=(10, 6))
bars = plt.bar(plot_df[age_col], plot_df[latest_col], color="#1f77b4")
plt.title(f"연령계층별 취업자 수 ({latest_col}, 전국)")
plt.xlabel("연령계층")
plt.ylabel("취업자 수")
plt.xticks(rotation=35, ha="right")

for b in bars:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width() / 2, h, f"{int(h):,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
output_path = ew_dir / "viz" / "employment_by_age_latest.png"
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=180)
print(f"Saved: {output_path}")
plt.show()
